In [1]:
print("Model evaluation notebook started")

Model evaluation notebook started


In [2]:
import pandas as pd
import evaluate
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [3]:
test_df = pd.read_csv("../data/test.csv")
test_df.head()

,Question,Answer,answer_length
0,what is zostavax for?- consider waiting for a ...,ZOSTAVAX® is a live attenuated virus vaccine i...,350
1,topiramate?,Topiramate is used alone or with other medicat...,707
2,where is tetracycline metabolized?,The majority of first-generation tetracyclines...,727
3,is lasix the only diuretic drug for fluid buil...,Commonly prescribed include: Furosemide (Lasix...,619
4,what spices can be used while on warfarin,We found 58 different plants that may alter th...,741


In [4]:
t5_path = "../models/t5-small-medicationqa-final"
flan_path = "../models/flan-t5-small-medicationqa-final"

t5_tokenizer = T5Tokenizer.from_pretrained(t5_path)
t5_model = T5ForConditionalGeneration.from_pretrained(t5_path)

flan_tokenizer = T5Tokenizer.from_pretrained(flan_path)
flan_model = T5ForConditionalGeneration.from_pretrained(flan_path)

print("Models loaded successfully")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Models loaded successfully


In [5]:
def generate_answer(model, tokenizer, question):
    input_text = "question: " + str(question)

    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).input_ids

    outputs = model.generate(
        input_ids,
        max_length=128,
        num_beams=4,
        repetition_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [6]:
sample_df = test_df.sample(30, random_state=42).reset_index(drop=True)

t5_predictions = []
flan_predictions = []

for question in sample_df["Question"]:
    t5_predictions.append(generate_answer(t5_model, t5_tokenizer, question))
    flan_predictions.append(generate_answer(flan_model, flan_tokenizer, question))

sample_df["T5_Prediction"] = t5_predictions
sample_df["FLAN_T5_Prediction"] = flan_predictions

sample_df.head()

,Question,Answer,answer_length,T5_Prediction,FLAN_T5_Prediction
0,"why no side affects on hydromorphone listed, y...",Hydromorphone may cause side effects.,37,"Hydromorphone is not listed as a side effect, ...",it is not a part of the list
1,what is zostavax for?- consider waiting for a ...,ZOSTAVAX® is a live attenuated virus vaccine i...,350,zostavax is a vaccine that can be used to trea...,Vaccine
2,can i take meloxicam along with tylenol + cod?,Tylenol (generic Acetaminophen) is commonly us...,507,meloxicam and tylenol can be used as a supplem...,no
3,what spices can be used while on warfarin,We found 58 different plants that may alter th...,741,"Warfarin is used to treat warfarin, which can ...",tequila
4,does marijuana use lead to negative health out...,"Marijuana can cause problems with memory, lear...",331,Marijuana is used as a drug that can be used t...,no


In [7]:
rouge = evaluate.load("rouge")

t5_rouge = rouge.compute(
    predictions=sample_df["T5_Prediction"].tolist(),
    references=sample_df["Answer"].tolist()
)

flan_rouge = rouge.compute(
    predictions=sample_df["FLAN_T5_Prediction"].tolist(),
    references=sample_df["Answer"].tolist()
)

print("T5 ROUGE:", t5_rouge)
print("FLAN-T5 ROUGE:", flan_rouge)

T5 ROUGE: {'rouge1': np.float64(0.11810288122758053), 'rouge2': np.float64(0.013870652707452677), 'rougeL': np.float64(0.09053136202108551), 'rougeLsum': np.float64(0.09069405731014973)}
FLAN-T5 ROUGE: {'rouge1': np.float64(0.019929610756948886), 'rouge2': np.float64(0.0030394265232974913), 'rougeL': np.float64(0.017774199687868752), 'rougeLsum': np.float64(0.018374658115665313)}


In [8]:
bleu = evaluate.load("bleu")

t5_bleu = bleu.compute(
    predictions=sample_df["T5_Prediction"].tolist(),
    references=[[ref] for ref in sample_df["Answer"].tolist()]
)

flan_bleu = bleu.compute(
    predictions=sample_df["FLAN_T5_Prediction"].tolist(),
    references=[[ref] for ref in sample_df["Answer"].tolist()]
)

print("T5 BLEU:", t5_bleu)
print("FLAN-T5 BLEU:", flan_bleu)

T5 BLEU: {'bleu': 0.0, 'precisions': [0.3164251207729469, 0.0390625, 0.002824858757062147, 0.0], 'brevity_penalty': 0.021429320688650295, 'length_ratio': 0.20648379052369079, 'translation_length': 414, 'reference_length': 2005}
FLAN-T5 BLEU: {'bleu': 0.0, 'precisions': [0.2328767123287671, 0.023255813953488372, 0.0, 0.0], 'brevity_penalty': 3.2067811956504384e-12, 'length_ratio': 0.03640897755610972, 'translation_length': 73, 'reference_length': 2005}


In [10]:
results = pd.DataFrame({
    "Model": ["T5-small", "FLAN-T5-small"],
    "ROUGE-1": [t5_rouge["rouge1"], flan_rouge["rouge1"]],
    "ROUGE-2": [t5_rouge["rouge2"], flan_rouge["rouge2"]],
    "ROUGE-L": [t5_rouge["rougeL"], flan_rouge["rougeL"]],
    "BLEU": [t5_bleu["bleu"], flan_bleu["bleu"]]
})

results

,Model,ROUGE-1,ROUGE-2,ROUGE-L,BLEU
0,T5-small,0.118103,0.013871,0.090531,0.0
1,FLAN-T5-small,0.019930,0.003039,0.017774,0.0


In [11]:
results.to_csv("../reports/model_evaluation_results.csv", index=False)
sample_df.to_csv("../reports/model_predictions_sample.csv", index=False)

print("Evaluation results saved successfully.")

Evaluation results saved successfully.
